<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/MAIN_OMIWORK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
# ============================================================
# ECGFOUNDER — MODEL SETUP
# CELL 1: DOWNLOAD PRETRAINED 12-LEAD MODEL
# ============================================================

import os
import requests
from tqdm.auto import tqdm
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "ECGFounder"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "12_lead_ECGFounder.pth"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# Official Hugging Face download URL
MODEL_URL = (
    "https://huggingface.co/PKUDigitalHealth/ECGFounder/"
    "resolve/main/12_lead_ECGFounder.pth"
)

# ------------------------------------------------------------
# DOWNLOAD
# ------------------------------------------------------------

# if os.path.exists(MODEL_PATH):

#     print("ECGFounder model already exists.")
#     print("Path:", MODEL_PATH)

# else:

#     print("Downloading ECGFounder...")
#     print("Size: approximately 370 MB")
#     print()

#     response = requests.get(
#         MODEL_URL,
#         stream=True
#     )

#     response.raise_for_status()

#     total_size = int(
#         response.headers.get("content-length", 0)
#     )

#     with open(MODEL_PATH, "wb") as f:

#         with tqdm(
#             total=total_size,
#             unit="B",
#             unit_scale=True,
#             desc="ECGFounder"
#         ) as pbar:

#             for chunk in response.iter_content(
#                 chunk_size=1024 * 1024
#             ):

#                 if chunk:

#                     f.write(chunk)
#                     pbar.update(len(chunk))

#     print("\nDownload complete.")

# # ------------------------------------------------------------
# # VERIFY FILE
# # ------------------------------------------------------------

# file_size_mb = (
#     os.path.getsize(MODEL_PATH)
#     / (1024 ** 2)
# )

# print("\n" + "=" * 60)
# print("ECGFOUNDER DOWNLOAD CHECK")
# print("=" * 60)

# print("File:")
# print(MODEL_PATH)

# print(f"\nSize: {file_size_mb:.2f} MB")

# print(
#     "Exists:",
#     os.path.exists(MODEL_PATH)
# )

# print("=" * 60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [57]:
# ============================================================
# ECGFOUNDER — CELL 2
# INSPECT PRETRAINED CHECKPOINT
# ============================================================

import torch
import os

MODEL_PATH = (
    "/content/drive/MyDrive/ECG_Project/"
    "ECGFounder/12_lead_ECGFounder.pth"
)

print("=" * 70)
print("LOADING ECGFOUNDER CHECKPOINT")
print("=" * 70)

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:")
print(type(checkpoint))

# ------------------------------------------------------------
# INSPECT TOP-LEVEL CONTENT
# ------------------------------------------------------------

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")

    for key in checkpoint.keys():
        print(" -", key)

else:

    print("\nCheckpoint is not a dictionary.")

# ------------------------------------------------------------
# BASIC INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHECKPOINT INSPECTION COMPLETE")
print("=" * 70)

LOADING ECGFOUNDER CHECKPOINT

Checkpoint type:
<class 'dict'>

Checkpoint keys:
 - epoch
 - step
 - state_dict
 - optimizer
 - scheduler
 - val_auroc

CHECKPOINT INSPECTION COMPLETE


In [58]:
# ============================================================
# ECGFOUNDER — CELL 3
# INSPECT MODEL WEIGHTS
# ============================================================

state_dict = checkpoint["state_dict"]

print("=" * 70)
print("ECGFOUNDER STATE DICTIONARY")
print("=" * 70)

print("\nNumber of parameter tensors:", len(state_dict))

print("\nFirst 30 parameter names:\n")

for i, (name, tensor) in enumerate(state_dict.items()):

    print(
        f"{i+1:2d}. "
        f"{name:<70} "
        f"shape={tuple(tensor.shape)}"
    )

    if i == 29:
        break



# ============================================================
# ECGFOUNDER — CELL 4
# INSPECT FINAL MODEL LAYERS
# ============================================================

print("=" * 70)
print("LAST 40 ECGFOUNDER PARAMETERS")
print("=" * 70)

items = list(state_dict.items())

for i, (name, tensor) in enumerate(items[-40:], start=len(items) - 39):

    print(
        f"{i:3d}. "
        f"{name:<70} "
        f"shape={tuple(tensor.shape)}"
    )


ECGFOUNDER STATE DICTIONARY

Number of parameter tensors: 509

First 30 parameter names:

 1. first_conv.conv.weight                                                 shape=(64, 12, 16)
 2. first_conv.conv.bias                                                   shape=(64,)
 3. first_bn.weight                                                        shape=(64,)
 4. first_bn.bias                                                          shape=(64,)
 5. first_bn.running_mean                                                  shape=(64,)
 6. first_bn.running_var                                                   shape=(64,)
 7. first_bn.num_batches_tracked                                           shape=()
 8. stage_list.0.block_list.0.bn1.weight                                   shape=(64,)
 9. stage_list.0.block_list.0.bn1.bias                                     shape=(64,)
10. stage_list.0.block_list.0.bn1.running_mean                             shape=(64,)
11. stage_list.0.block_list.0.bn1.ru

In [59]:
import torch
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = "/content/drive/MyDrive/ECG_Project/ECGFounder/12_lead_ECGFounder.pth"

import numpy as np
from collections import Counter
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, data, label):
        self.data = data
        self.label = label

    def __getitem__(self, index):
        return (torch.tensor(self.data[index], dtype=torch.float), torch.tensor(self.label[index], dtype=torch.long))

    def __len__(self):
        return len(self.data)

class MyConv1dPadSame(nn.Module):
    """
    extend nn.Conv1d to support SAME padding

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super(MyConv1dPadSame, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups = groups
        self.conv = torch.nn.Conv1d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

    def forward(self, x):

        net = x

        # compute pad shape
        in_dim = net.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)
        # print(net.shape)
        net = self.conv(net)

        return net

class MyMaxPool1dPadSame(nn.Module):
    """
    extend nn.MaxPool1d to support SAME padding

    params:
        kernel_size: kernel size
        stride: the stride of the window. Default value is kernel_size

    input: (n_sample, n_channel, n_length)
    """
    def __init__(self, kernel_size):
        super(MyMaxPool1dPadSame, self).__init__()
        self.kernel_size = kernel_size
        self.max_pool = torch.nn.MaxPool1d(kernel_size=self.kernel_size)

    def forward(self, x):

        net = x

        # compute pad shape
        p = max(0, self.kernel_size - 1)
        pad_left = p // 2
        pad_right = p - pad_left
        net = F.pad(net, (pad_left, pad_right), "constant", 0)

        net = self.max_pool(net)

        return net

class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

class BasicBlock(nn.Module):
    """
    Basic Block:
        conv1 -> convk -> conv1

    params:
        in_channels: number of input channels
        out_channels: number of output channels
        ratio: ratio of channels to out_channels
        kernel_size: kernel window length
        stride: kernel step size
        groups: number of groups in convk
        downsample: whether downsample length
        use_bn: whether use batch_norm
        use_do: whether use dropout

    input: (n_sample, in_channels, n_length)
    output: (n_sample, out_channels, (n_length+stride-1)//stride)
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, downsample, is_first_block=False, use_bn=True, use_do=True):
        super(BasicBlock, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.downsample = downsample
        self.stride = stride if self.downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        self.middle_channels = int(self.out_channels * self.ratio)

        # the first conv, conv1
        self.bn1 = nn.BatchNorm1d(in_channels)
        self.activation1 = Swish()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(
            in_channels=self.in_channels,
            out_channels=self.middle_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # the second conv, convk
        self.bn2 = nn.BatchNorm1d(self.middle_channels)
        self.activation2 = Swish()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.middle_channels,
            kernel_size=self.kernel_size,
            stride=self.stride,
            groups=self.groups)

        # the third conv, conv1
        self.bn3 = nn.BatchNorm1d(self.middle_channels)
        self.activation3 = Swish()
        self.do3 = nn.Dropout(p=0.5)
        self.conv3 = MyConv1dPadSame(
            in_channels=self.middle_channels,
            out_channels=self.out_channels,
            kernel_size=1,
            stride=1,
            groups=1)

        # Squeeze-and-Excitation
        r = 2
        self.se_fc1 = nn.Linear(self.out_channels, self.out_channels//r)
        self.se_fc2 = nn.Linear(self.out_channels//r, self.out_channels)
        self.se_activation = Swish()

        if self.downsample:
            self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):

        identity = x

        out = x
        # the first conv, conv1
        if not self.is_first_block:
            if self.use_bn:
                out = self.bn1(out)
            out = self.activation1(out)
            if self.use_do:
                out = self.do1(out)
        out = self.conv1(out)

        # the second conv, convk
        if self.use_bn:
            out = self.bn2(out)
        out = self.activation2(out)
        if self.use_do:
            out = self.do2(out)
        out = self.conv2(out)

        # the third conv, conv1
        if self.use_bn:
            out = self.bn3(out)
        out = self.activation3(out)
        if self.use_do:
            out = self.do3(out)
        out = self.conv3(out) # (n_sample, n_channel, n_length)

        # Squeeze-and-Excitation
        se = out.mean(-1) # (n_sample, n_channel)
        se = self.se_fc1(se)
        se = self.se_activation(se)
        se = self.se_fc2(se)
        se = torch.sigmoid(se) # (n_sample, n_channel)
        out = torch.einsum('abc,ab->abc', out, se)

        # if downsample, also downsample identity
        if self.downsample:
            identity = self.max_pool(identity)

        # if expand channel, also pad zeros to identity
        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1,-2)
            ch1 = (self.out_channels-self.in_channels)//2
            ch2 = self.out_channels-self.in_channels-ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1,-2)

        # shortcut
        out += identity

        return out

class BasicStage(nn.Module):
    """
    Basic Stage:
        block_1 -> block_2 -> ... -> block_M
    """
    def __init__(self, in_channels, out_channels, ratio, kernel_size, stride, groups, i_stage, m_blocks, use_bn=True, use_do=True, verbose=False):
        super(BasicStage, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.groups = groups
        self.i_stage = i_stage
        self.m_blocks = m_blocks
        self.use_bn = use_bn
        self.use_do = use_do
        self.verbose = verbose

        self.block_list = nn.ModuleList()
        for i_block in range(self.m_blocks):

            # first block
            if self.i_stage == 0 and i_block == 0:
                self.is_first_block = True
            else:
                self.is_first_block = False
            # downsample, stride, input
            if i_block == 0:
                self.downsample = True
                self.stride = stride
                self.tmp_in_channels = self.in_channels
            else:
                self.downsample = False
                self.stride = 1
                self.tmp_in_channels = self.out_channels

            # build block
            tmp_block = BasicBlock(
                in_channels=self.tmp_in_channels,
                out_channels=self.out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=self.groups,
                downsample=self.downsample,
                is_first_block=self.is_first_block,
                use_bn=self.use_bn,
                use_do=self.use_do)
            self.block_list.append(tmp_block)

    def forward(self, x):

        out = x

        for i_block in range(self.m_blocks):
            net = self.block_list[i_block]
            out = net(out)
            if self.verbose:
                print('stage: {}, block: {}, in_channels: {}, out_channels: {}, outshape: {}'.format(self.i_stage, i_block, net.in_channels, net.out_channels, list(out.shape)))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv1.in_channels, net.conv1.out_channels, net.conv1.kernel_size, net.conv1.stride, net.conv1.groups))
                print('stage: {}, block: {}, convk: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv2.in_channels, net.conv2.out_channels, net.conv2.kernel_size, net.conv2.stride, net.conv2.groups))
                print('stage: {}, block: {}, conv1: {}->{} k={} s={} C={}'.format(self.i_stage, i_block, net.conv3.in_channels, net.conv3.out_channels, net.conv3.kernel_size, net.conv3.stride, net.conv3.groups))

        return out

class Net1D(nn.Module):
    """

    Input:
        X: (n_samples, n_channel, n_length)
        Y: (n_samples)

    Output:
        out: (n_samples)

    params:
        in_channels
        base_filters
        filter_list: list, filters for each stage
        m_blocks_list: list, number of blocks of each stage
        kernel_size
        stride
        groups_width
        n_stages
        n_classes
        use_bn
        use_do

    """

    def __init__(self, in_channels, base_filters, ratio, filter_list, m_blocks_list, kernel_size, stride, groups_width, n_classes, use_bn=True, use_do=True, return_features=False, verbose=False):
        super(Net1D, self).__init__()

        self.in_channels = in_channels
        self.base_filters = base_filters
        self.ratio = ratio
        self.filter_list = filter_list
        self.m_blocks_list = m_blocks_list
        self.kernel_size = kernel_size
        self.stride = stride
        self.groups_width = groups_width
        self.n_stages = len(filter_list)
        self.n_classes = n_classes
        self.use_bn = use_bn
        self.use_do = use_do
        self.return_features = return_features
        self.verbose = verbose

        # first conv
        self.first_conv = MyConv1dPadSame(
            in_channels=in_channels,
            out_channels=self.base_filters,
            kernel_size=self.kernel_size,
            stride=2)
        self.first_bn = nn.BatchNorm1d(base_filters)
        self.first_activation = Swish()

        # stages
        self.stage_list = nn.ModuleList()
        in_channels = self.base_filters
        for i_stage in range(self.n_stages):

            out_channels = self.filter_list[i_stage]
            m_blocks = self.m_blocks_list[i_stage]
            tmp_stage = BasicStage(
                in_channels=in_channels,
                out_channels=out_channels,
                ratio=self.ratio,
                kernel_size=self.kernel_size,
                stride=self.stride,
                groups=out_channels//self.groups_width,
                i_stage=i_stage,
                m_blocks=m_blocks,
                use_bn=self.use_bn,
                use_do=self.use_do,
                verbose=self.verbose)
            self.stage_list.append(tmp_stage)
            in_channels = out_channels

        # final prediction
        self.dense = nn.Linear(in_channels, n_classes)

    def forward(self, x):

        out = x

        # first conv
        out = self.first_conv(out)
        if self.use_bn:
            out = self.first_bn(out)
        out = self.first_activation(out)

        # stages
        for i_stage in range(self.n_stages):
            net = self.stage_list[i_stage]
            out = net(out)

        # final prediction
        deep_features = out.mean(-1)
        out = self.dense(deep_features)

        if self.return_features:
            return out, deep_features
        else:
            return out

if __name__ == "__main__":
  model = Net1D(
      in_channels=12,
      base_filters=64, #32 64
      ratio=1,
      filter_list=[64,160,160,400,400,1024,1024],    #[16,32,32,80,80,256,256] [32,64,64,160,160,512,512] [64,160,160,400,400,1024,1024]
      m_blocks_list=[2,2,2,3,3,4,4],   #[2,2,2,2,2,2,2] [2,2,2,3,3,4,4]
      kernel_size=16,
      stride=2,
      groups_width=16,
      verbose=True,
      use_bn=False,
      use_do=False,
      n_classes=150)
  print(model)



Net1D(
  (first_conv): MyConv1dPadSame(
    (conv): Conv1d(12, 64, kernel_size=(16,), stride=(2,))
  )
  (first_bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (first_activation): Swish()
  (stage_list): ModuleList(
    (0): BasicStage(
      (block_list): ModuleList(
        (0): BasicBlock(
          (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation1): Swish()
          (do1): Dropout(p=0.5, inplace=False)
          (conv1): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
          )
          (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation2): Swish()
          (do2): Dropout(p=0.5, inplace=False)
          (conv2): MyConv1dPadSame(
            (conv): Conv1d(64, 64, kernel_size=(16,), stride=(2,), groups=4)
          )
          (bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [60]:
# Build exact ECGFounder architecture
model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=150,
    use_bn=False,
    use_do=False,
    return_features=True,
    verbose=False
)

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["state_dict"])

model = model.to(DEVICE)

print("✓ ECGFounder loaded successfully")
print("Device:", DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

✓ ECGFounder loaded successfully
Device: cuda
Parameters: 30819014


In [61]:
# ============================================================
# CELL 5 — CHANGE ECGFOUNDER HEAD FOR OMI
# ============================================================

import torch.nn as nn

# ECGFounder final embedding size = 1024
model.dense = nn.Linear(1024, 1)

# We only want prediction during training
model.return_features = False

model = model.to(DEVICE)

print("New classification head:")
print(model.dense)

print("\nOutput classes: 1")
print("Task: OMI vs non-OMI")

New classification head:
Linear(in_features=1024, out_features=1, bias=True)

Output classes: 1
Task: OMI vs non-OMI


In [62]:
# ============================================================
# CELL 6 — H5 DATASET FOR OMI
# ============================================================

import h5py
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

BASE = "/content/drive/MyDrive/ECG_Project/results/preprocessing/processed"

TRAIN_H5  = f"{BASE}/train_fixed.h5"
VAL_H5    = f"{BASE}/validation.h5"

TRAIN_CSV = f"{BASE}/train_omi.csv"
VAL_CSV   = f"{BASE}/validation_omi.csv"


class OMIDataset(Dataset):

    def __init__(self, h5_path, csv_path):

        self.h5_path = h5_path
        self.df = pd.read_csv(csv_path)

        # Read IDs once for verification
        with h5py.File(self.h5_path, "r") as f:

            ids = f["record_ids"][:]

            self.record_ids = [
                x.decode() if isinstance(x, bytes) else str(x)
                for x in ids
            ]

            self.length = len(f["signals"])

        # Safety checks
        assert self.length == len(self.df), (
            f"H5={self.length}, CSV={len(self.df)}"
        )

        csv_ids = self.df["record_id"].astype(str).str.zfill(5).tolist()
        h5_ids  = [str(x).zfill(5) for x in self.record_ids]

        assert csv_ids == h5_ids, "CSV and H5 record order mismatch!"

        self.labels = self.df["OMI"].astype(np.float32).values

        self.h5 = None

        print(
            f"Loaded {self.length} ECGs | "
            f"OMI={int(self.labels.sum())} | "
            f"non-OMI={int(len(self.labels)-self.labels.sum())}"
        )


    def __len__(self):
        return self.length


    def __getitem__(self, idx):

        # Open H5 lazily
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, "r")

        ecg = self.h5["signals"][idx]

        ecg = torch.tensor(
            ecg,
            dtype=torch.float32
        )

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        return ecg, label

In [63]:
# ============================================================
# CELL 7 — DATALOADERS
# ============================================================

train_dataset = OMIDataset(
    TRAIN_H5,
    TRAIN_CSV
)

val_dataset = OMIDataset(
    VAL_H5,
    VAL_CSV
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Loaded 15273 ECGs | OMI=985 | non-OMI=14288
Loaded 2685 ECGs | OMI=166 | non-OMI=2519

Train batches: 955
Validation batches: 168


In [64]:
# ============================================================
# CELL 8 — VERIFY INPUT
# ============================================================

x, y = next(iter(train_loader))

print("ECG batch:", x.shape)
print("Labels:", y.shape)

print("ECG dtype:", x.dtype)
print("Label dtype:", y.dtype)

print("Example labels:")
print(y[:20])

ECG batch: torch.Size([16, 12, 5000])
Labels: torch.Size([16])
ECG dtype: torch.float32
Label dtype: torch.float32
Example labels:
tensor([0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])


In [65]:
# ============================================================
# CELL 9 — EXPERIMENT CONFIGURATION
# ============================================================

import os
import torch
import torch.nn as nn
import numpy as np

# ------------------------------------------------------------
#  EXPERIMENT SETTINGS
# ------------------------------------------------------------

CONFIG = {
    "unfreeze_mode": "last2",

    "head_lr": 1e-4,
    "backbone_lr": 1e-5,

    "weight_decay": 1e-4,

    "epochs": 30,
    "patience": 5,

    "max_grad_norm": 5.0,

    "save_dir":
        "/content/drive/MyDrive/ECG_Project/OMI_model/task1_baseline_last2"
}

In [66]:
# ============================================================
# CELL 10 — AUTOMATIC CLASS WEIGHT
# ============================================================

n_pos = np.sum(train_dataset.labels == 1)
n_neg = np.sum(train_dataset.labels == 0)

pos_weight_value = n_neg / n_pos

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("Class distribution")
print("------------------")
print("non-OMI:", n_neg)
print("OMI:", n_pos)
print("Positive class weight:", round(float(pos_weight_value), 4))

Class distribution
------------------
non-OMI: 14288
OMI: 985
Positive class weight: 14.5056


In [67]:
# ============================================================
# CELL 11 — REUSABLE FREEZE / UNFREEZE FUNCTION
# ============================================================

def configure_finetuning(model, mode="last2"):

    """
    Available modes:

    head
        -> only classification head trains

    last1
        -> head + final ECGFounder stage

    last2
        -> head + final 2 stages

    last3
        -> head + final 3 stages

    full
        -> entire ECGFounder trains
    """

    # --------------------------------------------------------
    # STEP 1 — freeze everything
    # --------------------------------------------------------

    for param in model.parameters():
        param.requires_grad = False

    # --------------------------------------------------------
    # STEP 2 — always train new OMI head
    # --------------------------------------------------------

    for param in model.dense.parameters():
        param.requires_grad = True

    # --------------------------------------------------------
    # STEP 3 — selectively unfreeze backbone
    # --------------------------------------------------------

    if mode == "head":

        pass

    elif mode == "last1":

        for param in model.stage_list[-1].parameters():
            param.requires_grad = True

    elif mode == "last2":

        for stage in model.stage_list[-2:]:
            for param in stage.parameters():
                param.requires_grad = True

    elif mode == "last3":

        for stage in model.stage_list[-3:]:
            for param in stage.parameters():
                param.requires_grad = True

    elif mode == "full":

        for param in model.parameters():
            param.requires_grad = True

    else:

        raise ValueError(
            "mode must be one of: "
            "'head', 'last1', 'last2', 'last3', 'full'"
        )

    return model

In [68]:
# ============================================================
# CELL 12 — APPLY FINE-TUNING STRATEGY
# ============================================================

model = configure_finetuning(
    model,
    CONFIG["unfreeze_mode"]
)

print(
    "Fine-tuning mode:",
    CONFIG["unfreeze_mode"]
)

print("\nTrainable components:\n")

for name, param in model.named_parameters():

    if param.requires_grad:
        print(name)

Fine-tuning mode: last2

Trainable components:

stage_list.5.block_list.0.bn1.weight
stage_list.5.block_list.0.bn1.bias
stage_list.5.block_list.0.conv1.conv.weight
stage_list.5.block_list.0.conv1.conv.bias
stage_list.5.block_list.0.bn2.weight
stage_list.5.block_list.0.bn2.bias
stage_list.5.block_list.0.conv2.conv.weight
stage_list.5.block_list.0.conv2.conv.bias
stage_list.5.block_list.0.bn3.weight
stage_list.5.block_list.0.bn3.bias
stage_list.5.block_list.0.conv3.conv.weight
stage_list.5.block_list.0.conv3.conv.bias
stage_list.5.block_list.0.se_fc1.weight
stage_list.5.block_list.0.se_fc1.bias
stage_list.5.block_list.0.se_fc2.weight
stage_list.5.block_list.0.se_fc2.bias
stage_list.5.block_list.1.bn1.weight
stage_list.5.block_list.1.bn1.bias
stage_list.5.block_list.1.conv1.conv.weight
stage_list.5.block_list.1.conv1.conv.bias
stage_list.5.block_list.1.bn2.weight
stage_list.5.block_list.1.bn2.bias
stage_list.5.block_list.1.conv2.conv.weight
stage_list.5.block_list.1.conv2.conv.bias
stage_

In [69]:
os.makedirs(
    CONFIG["save_dir"],
    exist_ok=True
)

In [70]:
# ============================================================
# CELL 13 — PARAMETER CHECK
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = (
    total_params - trainable_params
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.2f}%"
)

Total parameters:     30,666,289
Trainable parameters: 26,709,793
Frozen parameters:    3,956,496
Trainable percentage: 87.10%


In [71]:
def build_optimizer(model, config):

    head_params = []
    backbone_params = []

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if name.startswith("dense"):
            head_params.append(param)

        else:
            backbone_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": backbone_params,
                "lr": config["backbone_lr"]
            },
            {
                "params": head_params,
                "lr": config["head_lr"]
            }
        ],
        weight_decay=config["weight_decay"]
    )

    return optimizer


# IMPORTANT — actually create it
optimizer = build_optimizer(
    model,
    CONFIG
)

print("Optimizer created.")

for i, group in enumerate(optimizer.param_groups):

    n_params = sum(
        p.numel()
        for p in group["params"]
    )

    print(
        f"Group {i}: "
        f"LR={group['lr']:.1e} | "
        f"Parameters={n_params:,}"
    )

Optimizer created.
Group 0: LR=1.0e-05 | Parameters=26,708,768
Group 1: LR=1.0e-04 | Parameters=1,025


In [72]:
# ============================================================
# CELL 15 — VALIDATION FUNCTION
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix
)

def evaluate_model(
    model,
    loader,
    criterion,
    device,
    threshold=0.5
):

    model.eval()

    losses = []
    probabilities = []
    labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            logits = model(x).squeeze(1)

            loss = criterion(
                logits,
                y
            )

            probs = torch.sigmoid(
                logits
            )

            losses.append(
                loss.item()
            )

            probabilities.extend(
                probs.cpu().numpy()
            )

            labels.extend(
                y.cpu().numpy()
            )

    probabilities = np.asarray(
        probabilities
    )

    labels = np.asarray(
        labels
    ).astype(int)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    auroc = roc_auc_score(
        labels,
        probabilities
    )

    auprc = average_precision_score(
        labels,
        probabilities
    )

    f1 = f1_score(
        labels,
        predictions,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    sensitivity = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0
    )

    return {

        "loss": np.mean(losses),

        "auroc": auroc,

        "auprc": auprc,

        "f1": f1,

        "sensitivity": sensitivity,

        "specificity": specificity,

        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,

        # save these because later
        # we'll optimize threshold
        "probabilities": probabilities,
        "labels": labels
    }

In [73]:
# ============================================================
# CELL 16 — REUSABLE TRAINING FUNCTION
# ============================================================

from tqdm.auto import tqdm

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    config
):

    best_auprc = -1
    epochs_without_improvement = 0

    history = []

    mode = config["unfreeze_mode"]

    save_path = os.path.join(
        config["save_dir"],
        f"task1_ecgfounder_omi_{mode}.pth"
    )

    for epoch in range(
        1,
        config["epochs"] + 1
    ):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        train_losses = []

        train_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch:02d}/{config['epochs']} - Training",
            leave=False
        )

        for x, y in train_bar:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(x).squeeze(1)

            loss = criterion(
                logits,
                y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config["max_grad_norm"]
            )

            optimizer.step()

            train_losses.append(
                loss.item()
            )

            # Show current + average loss
            train_bar.set_postfix(
                current_loss=f"{loss.item():.4f}",
                avg_loss=f"{np.mean(train_losses):.4f}"
            )

        train_loss = np.mean(
            train_losses
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        metrics = evaluate_model(
            model,
            val_loader,
            criterion,
            device
        )

        epoch_results = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": metrics["loss"],
            "auroc": metrics["auroc"],
            "auprc": metrics["auprc"],
            "f1": metrics["f1"],
            "sensitivity": metrics["sensitivity"],
            "specificity": metrics["specificity"]
        }

        history.append(
            epoch_results
        )

        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print("\n" + "=" * 65)

        print(f"Mode: {mode}")

        print(
            f"Epoch {epoch:02d}/"
            f"{config['epochs']}"
        )

        print(
            f"Train Loss : {train_loss:.5f}"
        )

        print(
            f"Val Loss   : {metrics['loss']:.5f}"
        )

        print(
            f"AUROC      : {metrics['auroc']:.4f}"
        )

        print(
            f"AUPRC      : {metrics['auprc']:.4f}"
        )

        print(
            f"F1         : {metrics['f1']:.4f}"
        )

        print(
            f"Sensitivity: "
            f"{metrics['sensitivity']:.4f}"
        )

        print(
            f"Specificity: "
            f"{metrics['specificity']:.4f}"
        )

        print(
            f"TN={metrics['tn']} "
            f"FP={metrics['fp']} "
            f"FN={metrics['fn']} "
            f"TP={metrics['tp']}"
        )

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if metrics["auprc"] > best_auprc:

            best_auprc = metrics["auprc"]

            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "epoch":
                        epoch,

                    "mode":
                        mode,

                    "val_auprc":
                        metrics["auprc"],

                    "val_auroc":
                        metrics["auroc"],

                    "config":
                        config
                },

                save_path
            )

            print(
                "\n✓ New best model saved"
            )

            print(
                "Path:",
                save_path
            )

        else:

            epochs_without_improvement += 1

            print(
                f"\nNo improvement: "
                f"{epochs_without_improvement}/"
                f"{config['patience']}"
            )

            # ====================================================
            # EARLY STOPPING
            # ====================================================

            if (
                epochs_without_improvement
                >= config["patience"]
            ):

                print(
                    "\nEarly stopping triggered."
                )

                break

        print("=" * 65)

    # ============================================================
    # TRAINING COMPLETE
    # ============================================================

    print("\nTraining finished.")

    print(
        "Best validation AUPRC:",
        round(best_auprc, 4)
    )

    print(
        "Best checkpoint:",
        save_path
    )

    return history

In [74]:
# ============================================================
# CELL 17 — START EXPERIMENT
# ============================================================

history = train_model(
    model=model,

    train_loader=train_loader,

    val_loader=val_loader,

    optimizer=optimizer,

    criterion=criterion,

    device=DEVICE,

    config=CONFIG
)

Epoch 01/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


Mode: last2
Epoch 01/30
Train Loss : 0.96362
Val Loss   : 0.85549
AUROC      : 0.8625
AUPRC      : 0.3330
F1         : 0.3280
Sensitivity: 0.7470
Specificity: 0.8150
TN=2053 FP=466 FN=42 TP=124

✓ New best model saved
Path: /content/drive/MyDrive/ECG_Project/OMI_model/task1_baseline_last2/task1_ecgfounder_omi_last2.pth


Epoch 02/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


Mode: last2
Epoch 02/30
Train Loss : 0.80957
Val Loss   : 0.90039
AUROC      : 0.8694
AUPRC      : 0.3568
F1         : 0.3750
Sensitivity: 0.6867
Specificity: 0.8698
TN=2191 FP=328 FN=52 TP=114

✓ New best model saved
Path: /content/drive/MyDrive/ECG_Project/OMI_model/task1_baseline_last2/task1_ecgfounder_omi_last2.pth


Epoch 03/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]

...


Mode: last2
Epoch 10/30
Train Loss : 0.51559
Val Loss   : 0.99169
AUROC      : 0.8789
AUPRC      : 0.3713
F1         : 0.3936
Sensitivity: 0.6687
Specificity: 0.8861
TN=2232 FP=287 FN=55 TP=111

No improvement: 3/5


Epoch 11/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


Mode: last2
Epoch 11/30
Train Loss : 0.46700
Val Loss   : 1.15443
AUROC      : 0.8763
AUPRC      : 0.3701
F1         : 0.4145
Sensitivity: 0.5843
Specificity: 0.9186
TN=2314 FP=205 FN=69 TP=97

No improvement: 4/5


Epoch 12/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


Mode: last2
Epoch 12/30
Train Loss : 0.41517
Val Loss   : 1.23350
AUROC      : 0.8731
AUPRC      : 0.3668
F1         : 0.4286
Sensitivity: 0.5964
Specificity: 0.9218
TN=2322 FP=197 FN=67 TP=99

No improvement: 5/5

Early stopping triggered.

Training finished.
Best validation AUPRC: 0.392
Best checkpoint: /content/drive/MyDrive/ECG_Project/OMI_model/task1_baseline_last2/task1_ecgfounder_omi_last2.pth
